## 1. Loading a raw SpikeGLX recording with SpikeInterface
---

In [10]:
%matplotlib inline 
#Jupyter magic command so matplotlib plots render inline in the notebook instead of opening separate windows

import spikeinterface.full as si
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

spikeglx_folder = Path(r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\0_raw\20260522_r0000136_wt_1750\ephys\26_05_22_136_1_neuropixels_g0\26_05_22_136_1_neuropixels_g0_imec0")

stream_names, stream_ids = si.get_neo_streams('spikeglx', spikeglx_folder)
#SpikeInterface reads SpikeGLX files through the neo library. 
#This function scans the folder and lists which streams (data channels of the acquisition system) exist.
stream_names

raw_rec = si.read_spikeglx(spikeglx_folder, stream_name='imec0.ap')
raw_rec


SpikeGLXRecordingExtractor: 384 channels - 30.0kHz - 1 segments - 89,672,760 samples 
                            2,989.09s (49.82 minutes) - int16 dtype - 64.14 GiB

## 2. Filter and Remove Bad channels
---

In [12]:
# 1.BAND PASS FILTER: keeping only 1–300 Hz content  (LFP band)
rec1 = si.bandpass_filter(raw_rec, freq_min=1.0, freq_max=300, dtype="float32", ignore_low_freq_error=True) 

# 2.Detect bad channels (based on channel correlation with neighbors) that flags dead, noisy, or out-of-brain channels
#The function will give 2 outputs which channel (id) and what is the problem (label)
bad_channel_ids, channel_labels = si.detect_bad_channels(rec1)
# 3. remove those detected chanels 
rec2 = rec1.remove_channels(bad_channel_ids)
print('bad_channel_ids', bad_channel_ids)
# 4.Correct for delay between channels, so all channels are properly time-aligned
rec3 = si.phase_shift(rec2)

# 5. Common Median Reference (CMR) Subtracts the median signal across all channels from every channel, at each time point. 
# This removes noise that's shared across the whole probe (electrical noise picked up in common by all electrodes)
# leaving mostly the local signal.

#cmr_mean = si.common_reference(rec3, operator="average", reference="global")
#cmr_median = si.common_reference(rec3, operator="median", reference="global")

#6. remove powerline noise (50Hz)
rec4 = si.notch_filter(rec3, freq=50, q=30) # no CMR
#rec5 = si.notch_filter(cmr_mean, freq=50, q=30) #CMR (Mean)
#rec6 = si.notch_filter(rec4, freq=50, q=30) #CMR (Median)

signal = rec4
signal

c:\Users\ar687\anaconda3\envs\spikeinterface\Lib\site-packages\spikeinterface\preprocessing\filter.py:122: UserWarning: The margin size (150000 samples) is more than 20% of the global chunk size 30000 samples. This may lead to performance bottlenecks when chunking. Consider increasing the chunk_size or chunk_duration to minimize margin overhead.
  warnings.warn(


bad_channel_ids ['imec0.ap#AP138']


c:\Users\ar687\anaconda3\envs\spikeinterface\Lib\site-packages\spikeinterface\preprocessing\filter.py:122: UserWarning: The margin size (28647 samples) is more than 20% of the global chunk size 30000 samples. This may lead to performance bottlenecks when chunking. Consider increasing the chunk_size or chunk_duration to minimize margin overhead.
  warnings.warn(


NotchFilterRecording: 383 channels - 30.0kHz - 1 segments - 89,672,760 samples 
                      2,989.09s (49.82 minutes) - float32 dtype - 127.94 GiB

In [1]:
import scipy.signal as sig
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

ch = rec3.channel_ids[150]

fs = rec3.get_sampling_frequency()
trace = rec3.get_traces(channel_ids=[ch], return_scaled=True)[:, 0]

#############################################

#Visual check: PSD zoomed to 45–55 Hz
freqs, psd = sig.welch(trace, fs=fs, nperseg=int(fs*2))

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs, psd, color='black', linewidth=1)
ax.set_xlim(0, 200)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (µV²/Hz, log scale)')
ax.set_title(f'checking for line noise ')
ax.grid(alpha=0.3)

axins = inset_axes(ax, width="35%", height="40%", loc='upper right', borderpad=3)
freq_mask = (freqs >= 45) & (freqs <= 55)
axins.semilogy(freqs[freq_mask], psd[freq_mask], color='black', linewidth=1)
axins.set_xlim(45, 55)
axins.set_title('50 Hz zoom', fontsize=9)
axins.tick_params(labelsize=8)
axins.grid(alpha=0.3)
mark_inset(ax, axins, loc1=2, loc2=3, fc="none", ec="gray", linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

################################################

#Is the peak frequency stable over time?
results = []

for start_sec in [0, 300, 600]:  # adjust based on your recording length
    start_frame = int(start_sec * fs)
    end_frame = start_frame + int(fs * 60)  # 1-minute chunk
    chunk = rec3.get_traces(channel_ids=[ch], start_frame=start_frame, end_frame=end_frame, return_in_uV=True)[:, 0]
    f, p = sig.welch(chunk, fs=fs, nperseg=int(fs*2))

    mask = (f >= 45) & (f <= 55)
    f_window = f[mask]
    p_window = p[mask]

    peak_idx = np.argmax(p_window)
    peak_freq = f_window[peak_idx]
    peak_power = p_window[peak_idx]

    results.append({'start_sec': start_sec, 'peak_freq_hz': peak_freq, 'peak_power': peak_power})
    print(f"t={start_sec:>4}s | peak at {peak_freq:.2f} Hz | power = {peak_power:.4e} µV²/Hz")

################################################

#Power exactly at 50 Hz vs. its neighbors
target_freq = 50.0
freq_idx = np.argmin(np.abs(f - target_freq))  # index of the bin closest to 50 Hz

power_at_50 = p[freq_idx]
power_neighbors = np.mean([p[freq_idx-3], p[freq_idx-2], p[freq_idx+2], p[freq_idx+3]])
ratio = power_at_50 / power_neighbors

print(f"Power at 50 Hz: {power_at_50:.4e}")
print(f"Average power of nearby bins: {power_neighbors:.4e}")
print(f"Ratio: {ratio:.2f}x")

###############################################

#same check but across channels
for ch_test in [rec3.channel_ids[50], rec3.channel_ids[150], rec3.channel_ids[250]]:
    trace_test = rec3.get_traces(channel_ids=[ch_test], return_in_uV=True)[:, 0]
    f_test, p_test = sig.welch(trace_test, fs=fs, nperseg=int(fs*2))

    freq_idx = np.argmin(np.abs(f_test - 50.0))
    power_at_50 = p_test[freq_idx]
    power_neighbors = np.mean([p_test[freq_idx-3], p_test[freq_idx-2], p_test[freq_idx+2], p_test[freq_idx+3]])
    ratio = power_at_50 / power_neighbors

    print(f"Channel {ch_test}: ratio = {ratio:.2f}x")

##############################################

#check harmonics
for target_freq in [50.0, 100.0, 150.0]:
    freq_idx = np.argmin(np.abs(f_test - target_freq))
    power_at_target = p_test[freq_idx]
    power_neighbors = np.mean([p_test[freq_idx-3], p_test[freq_idx-2], p_test[freq_idx+2], p_test[freq_idx+3]])
    ratio = power_at_target / power_neighbors
    print(f"{target_freq} Hz: ratio = {ratio:.2f}x")

NameError: name 'rec3' is not defined

## 3. Downsampling 
---

> **Note:** `si.resample()` applies its own internal anti-aliasing low-pass filter before decimating. It uses a margin (default `margin_ms=100`) to avoid edge artifacts at chunk boundaries. This is safe even though the signal has already been bandpass-filtered, the additional filtering provides extra insurance against aliasing. 

In [13]:
Hz_1000 = si.resample(signal, resample_rate=1000) 

## 4. Saving Processed Recordings to Desk
___

`n_jobs = 4` 

> How many CPU processes to use in parallel while computing and writing. Since the pipeline (filtering, referencing, resampling) has to be actually run now, this can be slow for a full recording — n_jobs splits the work across multiple cores at once. n_jobs=4 means 4 worker processes running simultaneously. More cores → faster, but also more RAM used simultaneously (each worker holds its own chunk in memory). If you run into memory errors, lower this.

`chunk_duration="1s"`

> The recording is processed in chunks rather than all at once (which would blow up memory for a long recording at high sample rate). This tells SpikeInterface: process 1 second of data at a time, per chunk. Smaller chunks → less memory per worker but more overhead/coordination; larger chunks → more memory but slightly more efficient. "1s" is a very reasonable default — you generally don't need to touch this unless you hit memory limits (then go smaller, e.g. "0.5s") or want to slightly optimize speed on a machine with lots of RAM (go larger, e.g. "5s").

In [14]:
rec_folder = Path(r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\2_processed\20260522_r0000136_wt_1750\amir_data\down_sampled\1000_Hz")

rec_saved = Hz_1000.save(folder=rec_folder, n_jobs=4,chunk_duration="1s",progress_bar=True)

write_binary_recording 
engine=process - n_jobs=4 - samples_per_chunk=1,000 - chunk_memory=1.46 MiB - total_memory=5.84 MiB - chunk_duration=1.00s


write_binary_recording (workers: 4 processes spawn): 100%|██████████| 2990/2990 [1:21:18<00:00,  1.63s/it]


In [15]:
# 1. Load the saved 1000 Hz recording (without CAR)
rec_folder = Path(r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\2_processed\20260522_r0000136_wt_1750\amir_data\down_sampled\1000_Hz")
rec_1000 = si.load(rec_folder)

# 2. Apply CAR (mean)
rec_1000_car = si.common_reference(rec_1000, operator="average", reference="global")

# 3. Save to a NEW folder 
rec_folder_car = rec_folder.parent / "1000_Hz_CAR"
rec_saved_car = rec_1000_car.save(folder=rec_folder_car,n_jobs=4,chunk_duration="1s",progress_bar=True,     )

write_binary_recording 
engine=process - n_jobs=4 - samples_per_chunk=1,000 - chunk_memory=1.46 MiB - total_memory=5.84 MiB - chunk_duration=1.00s


write_binary_recording (workers: 4 processes spawn): 100%|██████████| 2990/2990 [01:17<00:00, 38.41it/s] 


In [ ]:
import os
print(os.getcwd())